In [7]:
import os, zipfile
import numpy as np
import tensorflow as tf
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

from google.colab import drive
drive.mount("/content/drive")

zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"
print("Extracted:", os.listdir(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracted: ['New hand pd Dataset', 'Hand Pd']


In [8]:
!pip -q install imagehash

In [9]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from collections import defaultdict, Counter
from PIL import Image
import imagehash

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

In [13]:
import imagehash
from PIL import Image
from collections import defaultdict

NEWHANDPD_ROOT = os.path.join(BASE, "New hand pd Dataset")
HEALTHY_ROOT = os.path.join(NEWHANDPD_ROOT, "Healthy")
PARK_ROOT = os.path.join(NEWHANDPD_ROOT, "Parkinson's")

image_paths_new = []
labels_new = []

def collect_images(root, label):
    for r, d, files in os.walk(root):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                image_paths_new.append(os.path.join(r, f))
                labels_new.append(label)

collect_images(HEALTHY_ROOT, 0)
collect_images(PARK_ROOT, 1)

print("NewHandPD total:", len(image_paths_new))
print("Healthy:", labels_new.count(0))
print("Parkinson:", labels_new.count(1))


def get_phash(img_path):
    try:
        img = Image.open(img_path).convert("L").resize((224,224))
        return imagehash.phash(img)
    except:
        return None


PHASH_DISTANCE_THRESHOLD = 4

hash_to_paths = defaultdict(list)

for p in tqdm(image_paths_new, desc="Hashing NewHandPD"):
    h = get_phash(p)
    if h is not None:
        hash_to_paths[str(h)].append(p)

hash_list = list(hash_to_paths.keys())
used = set()
final_keep_hashes = []

for i in tqdm(range(len(hash_list)), desc="Removing near duplicates NewHandPD"):
    if hash_list[i] in used:
        continue

    base_hash = imagehash.hex_to_hash(hash_list[i])
    used.add(hash_list[i])
    final_keep_hashes.append(hash_list[i])

    for j in range(i+1, len(hash_list)):
        if hash_list[j] in used:
            continue
        compare_hash = imagehash.hex_to_hash(hash_list[j])
        if base_hash - compare_hash <= PHASH_DISTANCE_THRESHOLD:
            used.add(hash_list[j])

path_to_label_new = dict(zip(image_paths_new, labels_new))

clean_paths_newhandpd = []
clean_labels_newhandpd = []

for h in final_keep_hashes:
    p = hash_to_paths[h][0]
    clean_paths_newhandpd.append(p)
    clean_labels_newhandpd.append(path_to_label_new[p])

print("\nCLEAN NEWHANDPD")
print("Images:", len(clean_paths_newhandpd))
print("Healthy:", clean_labels_newhandpd.count(0))
print("Parkinson:", clean_labels_newhandpd.count(1))


NewHandPD total: 512
Healthy: 233
Parkinson: 279


Removing near duplicates NewHandPD: 100%|██████████| 467/467 [00:02<00:00, 211.91it/s]


CLEAN NEWHANDPD
Images: 369
Healthy: 191
Parkinson: 178


In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

paths = np.array(clean_paths_newhandpd)
labels = np.array(clean_labels_newhandpd)

print("Total Clean:", len(paths))
print("Distribution:", Counter(labels))

Total Clean: 369
Distribution: Counter({np.int64(0): 191, np.int64(1): 178})


In [16]:
# 80% train, 20% temp
train_paths, temp_paths, y_train, y_temp = train_test_split(
    paths,
    labels,
    test_size=0.20,
    stratify=labels,
    random_state=42
)

# 10% val, 10% test
val_paths, test_paths, y_val, y_test = train_test_split(
    temp_paths,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", Counter(y_train))
print("Val:", Counter(y_val))
print("Test:", Counter(y_test))

Train: Counter({np.int64(0): 153, np.int64(1): 142})
Val: Counter({np.int64(0): 19, np.int64(1): 18})
Test: Counter({np.int64(0): 19, np.int64(1): 18})


In [17]:
print("Train-Val overlap:", len(set(train_paths) & set(val_paths)))
print("Train-Test overlap:", len(set(train_paths) & set(test_paths)))
print("Val-Test overlap:", len(set(val_paths) & set(test_paths)))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [18]:
current_train = len(train_paths)
target_train = 2000

multiply_factor = int((target_train / current_train) - 1)
print("Multiply factor:", multiply_factor)

Multiply factor: 5


In [19]:
AUG_ROOT = "/content/NewHandPD_Train_Expanded"
os.makedirs(AUG_ROOT, exist_ok=True)
os.makedirs(os.path.join(AUG_ROOT, "Healthy"), exist_ok=True)
os.makedirs(os.path.join(AUG_ROOT, "Parkinson"), exist_ok=True)

In [20]:
def augment_image(img):
    rows, cols = img.shape

    angle = np.random.uniform(-7, 7)
    M = cv2.getRotationMatrix2D((cols/2, rows/2), angle, 1)
    img = cv2.warpAffine(img, M, (cols, rows))

    if np.random.rand() > 0.5:
        img = cv2.flip(img, 1)

    scale = np.random.uniform(0.9, 1.1)
    M = cv2.getRotationMatrix2D((cols/2, rows/2), 0, scale)
    img = cv2.warpAffine(img, M, (cols, rows))

    alpha = np.random.uniform(0.9, 1.1)
    img = np.clip(img * alpha, 0, 255).astype(np.uint8)

    return img

In [21]:
MULTIPLY_FACTOR = 6

for path, label in tqdm(zip(train_paths, y_train), total=len(train_paths)):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    class_name = "Healthy" if label == 0 else "Parkinson"

    base = os.path.basename(path).split('.')[0]

    cv2.imwrite(os.path.join(AUG_ROOT, class_name, base + "_orig.png"), img)

    for i in range(MULTIPLY_FACTOR):
        aug = augment_image(img)
        cv2.imwrite(
            os.path.join(AUG_ROOT, class_name, base + f"_aug{i}.png"),
            aug
        )

100%|██████████| 295/295 [00:24<00:00, 12.09it/s]


In [22]:
expanded_train_paths = []
expanded_train_labels = []

for cls in os.listdir(AUG_ROOT):
    label = 0 if cls == "Healthy" else 1
    for r, d, files in os.walk(os.path.join(AUG_ROOT, cls)):
        for f in files:
            expanded_train_paths.append(os.path.join(r, f))
            expanded_train_labels.append(label)

print("Expanded Train Size:", len(expanded_train_paths))
print("Expanded Distribution:", Counter(expanded_train_labels))

Expanded Train Size: 2065
Expanded Distribution: Counter({0: 1071, 1: 994})


In [29]:
IMG_SIZE = 224
BATCH_SIZE = 16

def preprocess(path, label):
    img = cv2.imread(path.numpy().decode(), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, -1)

    label = np.float32(label.numpy())

    return img, label


def tf_preprocess(path, label):
    img, label = tf.py_function(
        preprocess,
        [path, label],
        [tf.float32, tf.float32]   # BOTH float32
    )
    img.set_shape((IMG_SIZE, IMG_SIZE, 1))
    label.set_shape(())
    return img, label

In [30]:
train_ds = make_ds(expanded_train_paths, expanded_train_labels, train=True)
val_ds   = make_ds(val_paths, y_val)
test_ds  = make_ds(test_paths, y_test)

In [31]:
def make_ds(paths, labels, train=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train:
        ds = ds.shuffle(2000)
    ds = ds.map(tf_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(expanded_train_paths, expanded_train_labels, train=True)
val_ds   = make_ds(val_paths, y_val)
test_ds  = make_ds(test_paths, y_test)

print("Datasets ready.")

Datasets ready.


In [32]:
DIMS = [526, 256, 128, 64, 32]

class PatchEmbedding(layers.Layer):
    def __init__(self, patch, dim):
        super().__init__()
        self.dim = dim
        self.proj = layers.Conv2D(dim, patch, strides=patch)

    def call(self, x):
        x = self.proj(x)
        x = tf.reshape(x, [tf.shape(x)[0], -1, self.dim])
        return x

class TransformerBlock(layers.Layer):
    def __init__(self, dim):
        super().__init__()
        self.norm1 = layers.LayerNormalization()
        self.attn = layers.MultiHeadAttention(2, max(dim//2,1))
        self.norm2 = layers.LayerNormalization()
        self.mlp = keras.Sequential([
            layers.Dense(dim*2, activation="gelu"),
            layers.Dense(dim)
        ])

    def call(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

def build_model():
    inp = layers.Input((224,224,1))
    x = PatchEmbedding(16, DIMS[0])(inp)

    for i in range(len(DIMS)):
        x = TransformerBlock(DIMS[i])(x)
        if i < len(DIMS)-1:
            x = layers.Dense(DIMS[i+1])(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(16, activation="relu")(x)
    x = layers.Dense(8, activation="relu")(x)
    x = layers.Dense(4, activation="relu")(x)
    x = layers.Dense(2, activation="relu")(x)

    out = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inp, out)

model = build_model()

In [33]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

In [34]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30
)

Epoch 1/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 70s 282ms/step - accuracy: 0.4916 - auc: 0.5000 - loss: 0.6931 - val_accuracy: 0.5135 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 2/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 101ms/step - accuracy: 0.5059 - auc: 0.5000 - loss: 0.6931 - val_accuracy: 0.5135 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 3/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - accuracy: 0.5195 - auc: 0.5000 - loss: 0.6931 - val_accuracy: 0.5135 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 4/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 20s 93ms/step - accuracy: 0.5282 - auc: 0.5000 - loss: 0.6931 - val_accuracy: 0.5135 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 5/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.5174 - auc: 0.5000 - loss: 0.6931 - val_accuracy: 0.5135 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 6/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - accuracy: 0.5035 - auc: 0.5000 - loss: 0.6931 - val_accuracy: 0.5135 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 7/30
130/130 ━━━━━━━

In [35]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, patch_size, strides=patch_size)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        num_patches = (input_shape[1] // 16) ** 2
        self.cls_token = self.add_weight(
            shape=(1, 1, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )
        self.pos_embed = self.add_weight(
            shape=(1, num_patches + 1, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

    def call(self, x):
        x = self.proj(x)
        x = tf.reshape(x, [tf.shape(x)[0], -1, self.embed_dim])

        batch = tf.shape(x)[0]
        cls = tf.repeat(self.cls_token, repeats=batch, axis=0)
        x = tf.concat([cls, x], axis=1)

        return x + self.pos_embed

In [36]:
def build_model():
    inputs = layers.Input((224,224,1))

    x = PatchEmbedding(16, 526)(inputs)

    dims = [526, 256, 128, 64, 32]

    for i in range(len(dims)):
        x = TransformerBlock(dims[i])(x)
        if i < len(dims)-1:
            x = layers.Dense(dims[i+1])(x)

    x = layers.LayerNormalization()(x)

    cls_token = x[:, 0]   # take CLS

    x = layers.Dense(16, activation="gelu")(cls_token)
    x = layers.Dense(8, activation="gelu")(x)
    x = layers.Dense(4, activation="gelu")(x)
    x = layers.Dense(2, activation="gelu")(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs)

model = build_model()

In [37]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

In [38]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30
)

Epoch 1/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 70s 272ms/step - accuracy: 0.4875 - auc: 0.4871 - loss: 0.6956 - val_accuracy: 0.4865 - val_auc: 0.5000 - val_loss: 0.6944
Epoch 2/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 101ms/step - accuracy: 0.4752 - auc: 0.4993 - loss: 0.6951 - val_accuracy: 0.4865 - val_auc: 0.5000 - val_loss: 0.6941
Epoch 3/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.4853 - auc: 0.4936 - loss: 0.6942 - val_accuracy: 0.4865 - val_auc: 0.5000 - val_loss: 0.6938
Epoch 4/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.4831 - auc: 0.4999 - loss: 0.6939 - val_accuracy: 0.4865 - val_auc: 0.5000 - val_loss: 0.6936
Epoch 5/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.4693 - auc: 0.5000 - loss: 0.6941 - val_accuracy: 0.4865 - val_auc: 0.5000 - val_loss: 0.6935
Epoch 6/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.4906 - auc: 0.5000 - loss: 0.6934 - val_accuracy: 0.4865 - val_auc: 0.5000 - val_loss: 0.6933
Epoch 7/30
130/130 ━━━━━━━

In [39]:
preds = model.predict(val_ds)
pred_labels = (preds > 0.5).astype(int)

print("Predicted class distribution:")
print(np.unique(pred_labels, return_counts=True))

print("True class distribution:")
print(np.unique(y_val, return_counts=True))

3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step
Predicted class distribution:
(array([0]), array([37]))
True class distribution:
(array([0, 1]), array([19, 18]))


In [40]:
inputs = layers.Input((224,224,1))
x = layers.Conv2D(16,3,activation="relu")(inputs)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(32,3,activation="relu")(x)
x = layers.MaxPooling2D()(x)
x = layers.Flatten()(x)
x = layers.Dense(64,activation="relu")(x)
outputs = layers.Dense(1,activation="sigmoid")(x)

test_model = keras.Model(inputs,outputs)

test_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

test_model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5
130/130 ━━━━━━━━━━━━━━━━━━━━ 18s 99ms/step - accuracy: 0.6469 - auc: 0.6874 - loss: 0.7035 - val_accuracy: 0.8378 - val_auc: 0.9327 - val_loss: 0.3584
Epoch 2/5
130/130 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.8427 - auc: 0.9331 - loss: 0.3329 - val_accuracy: 0.8378 - val_auc: 0.9430 - val_loss: 0.4026
Epoch 3/5
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 75ms/step - accuracy: 0.9088 - auc: 0.9726 - loss: 0.2147 - val_accuracy: 0.7568 - val_auc: 0.9591 - val_loss: 0.7113
Epoch 4/5
130/130 ━━━━━━━━━━━━━━━━━━━━ 11s 78ms/step - accuracy: 0.9165 - auc: 0.9819 - loss: 0.1843 - val_accuracy: 0.8378 - val_auc: 0.9518 - val_loss: 0.4747
Epoch 5/5
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 78ms/step - accuracy: 0.9629 - auc: 0.9941 - loss: 0.1146 - val_accuracy: 0.7838 - val_auc: 0.9035 - val_loss: 0.4464


In [41]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.proj = layers.Conv2D(embed_dim, patch_size, strides=patch_size)

    def build(self, input_shape):
        num_patches = (input_shape[1] // self.patch_size) ** 2

        self.cls_token = self.add_weight(
            shape=(1, 1, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

        self.pos_embed = self.add_weight(
            shape=(1, num_patches + 1, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

    def call(self, x):
        x = self.proj(x)
        x = tf.reshape(x, [tf.shape(x)[0], -1, self.embed_dim])

        batch = tf.shape(x)[0]
        cls = tf.repeat(self.cls_token, repeats=batch, axis=0)

        x = tf.concat([cls, x], axis=1)
        return x + self.pos_embed

In [42]:
class TransformerBlock(layers.Layer):
    def __init__(self, dim):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=4,
            key_dim=dim // 4
        )
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = keras.Sequential([
            layers.Dense(dim * 4, activation="gelu"),
            layers.Dense(dim)
        ])

    def call(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [44]:
def build_model():
    inputs = layers.Input((224,224,1))

    x = PatchEmbedding(4, 128)(inputs)

    dims = [128, 64, 32, 16]

    for i in range(len(dims)):
        x = TransformerBlock(dims[i])(x)
        if i < len(dims)-1:
            x = layers.Dense(dims[i+1])(x)

    x = layers.LayerNormalization()(x)

    cls_token = x[:, 0]

    x = layers.Dense(8, activation="gelu")(cls_token)
    x = layers.Dense(4, activation="gelu")(x)
    x = layers.Dense(2, activation="gelu")(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs)

model = build_model()

In [45]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30
)

Epoch 1/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.5104 - auc: 0.5078 - loss: 0.6932 - val_accuracy: 0.4865 - val_auc: 0.6827 - val_loss: 0.6941
Epoch 2/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 115s 885ms/step - accuracy: 0.4888 - auc: 0.5022 - loss: 0.6940 - val_accuracy: 0.6757 - val_auc: 0.6711 - val_loss: 0.6912
Epoch 3/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 143s 891ms/step - accuracy: 0.5356 - auc: 0.5491 - loss: 0.6887 - val_accuracy: 0.4865 - val_auc: 0.6126 - val_loss: 0.6728
Epoch 4/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 116s 892ms/step - accuracy: 0.5278 - auc: 0.5618 - loss: 0.6853 - val_accuracy: 0.5135 - val_auc: 0.6287 - val_loss: 0.6835
Epoch 5/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 116s 892ms/step - accuracy: 0.5572 - auc: 0.5992 - loss: 0.6751 - val_accuracy: 0.4595 - val_auc: 0.7427 - val_loss: 0.6126
Epoch 6/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 143s 898ms/step - accuracy: 0.6550 - auc: 0.7138 - loss: 0.6280 - val_accuracy: 0.7838 - val_auc: 0.8699 - val_loss: 0.4875
Epoch 7/30
130/130 